In [ ]:
import numpy as np
import ufl

from mpi4py import MPI

import dolfinx
import dolfinx.fem.petsc
import dolfinx.mesh
import basix.ufl
from petsc4py import PETSc



length, height = 3., 2.0
Nx, Ny = 2,2
domain = dolfinx.mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([0., 0.]), np.array([length, height])],
    [Nx, Ny],
    cell_type=dolfinx.mesh.CellType.quadrilateral,
)

def up_bottom_boundary(x):
    #on_left = np.isclose(x[0], knots1[0])
    on_bottom = np.isclose(x[1], 0)
    #on_right = np.isclose(x[0], knots1[-1])
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top

def all_around_boundary(x):
    on_left = np.isclose(x[0], 0)
    on_bottom = np.isclose(x[1], 0)
    on_right = np.isclose(x[0], length)
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top|on_left|on_right

dim = domain.topology.dim
print(f"Mesh topology dimension d={dim}.")

degree = 2
#shape = (dim,)  # this means we want a vector field of size `dim`
v_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree=degree,
    #lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(domain, v_elem)

u_sol = dolfinx.fem.Function(V, name="solution")

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

x = ufl.SpatialCoordinate(domain)
#u_exact = x[0]*(length-x[0])*(1.+x[1])
u_exact = x[1]*x[0]*(length-x[0])*(height-x[1])

f_mms = -ufl.div(ufl.grad(u_exact))
T_mms = ufl.inner(ufl.grad(u_exact),ufl.FacetNormal(domain))

facet_dim = domain.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(domain, facet_dim, all_around_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    domain,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)

custom_metadata = {"quadrature_degree": 3}
custom_dx = ufl.Measure("dx", domain=domain, metadata=custom_metadata)
custom_ds = ufl.Measure("ds", domain=domain, subdomain_data=boundary_tags, metadata=custom_metadata)
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * custom_dx
L = ufl.inner(f_mms, v) * custom_dx 
#L+= ufl.inner(T_mms, v)*custom_ds(1) 
def left(x):
    return np.isclose(x[0], 0.)

def right(x):
    return np.isclose(x[0], length)
def up(x):
    return np.isclose(x[1], height)
def bottom(x):
    return np.isclose(x[1], 0.)


left_dofs = dolfinx.fem.locate_dofs_geometrical(V, left)
right_dofs = dolfinx.fem.locate_dofs_geometrical(V, right)
up_dofs = dolfinx.fem.locate_dofs_geometrical(V, up)
down_dofs = dolfinx.fem.locate_dofs_geometrical(V, bottom)
zero_vec = PETSc.ScalarType(0)
bcs = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, right_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, up_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, down_dofs, V),
]
problem = dolfinx.fem.petsc.LinearProblem(
    a, L, u=u_sol, bcs=bcs,
    petsc_options_prefix="poisson",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps"}
)
problem.solve()

#b_legendre = problem.b

# Assuming u_sol is your numerical solution, and u_exact is your reference
# (u_exact must be a dolfinx.fem.Function or a UFL spatial expression)

# Define the error
e = u_sol - u_exact

# 1. L2 Error
error_L2_form = dolfinx.fem.form(ufl.inner(e, e) * custom_dx)
error_L2 = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_L2_form), op=MPI.SUM))

# 2. H1 Error
error_H1_form = dolfinx.fem.form((ufl.inner(e, e) + ufl.inner(ufl.grad(e), ufl.grad(e))) * custom_dx)
error_H1 = np.sqrt(domain.comm.allreduce(dolfinx.fem.assemble_scalar(error_H1_form), op=MPI.SUM))

print(f"L2 Error:     {error_L2:.2e}")
print(f"H1 Error:     {error_H1:.2e}")
print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")


## BSplines

In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx


from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
import numpy.typing as npt

def refine(knots: npt.ArrayLike, p: int, n_times: int=1)->npt.NDArray:
    """Given `knots`, returns its dyadic refinement with multiplicity `p+1`
    at the extremities."""
    knots: npt.NDArray[np.float_] = np.asarray(knots)
    mult_left: int = np.searchsorted(knots, knots[0], side='right')
    mult_right: int = len(knots) - np.searchsorted(knots, knots[-1], side='left')
    pad_left: int = max(0, p + 1 - mult_left)
    pad_right: int = max(0, p + 1 - mult_right)
    if pad_left > 0 or pad_right > 0:
        knots = np.concatenate((
            np.full(pad_left, knots[0], dtype=knots.dtype),
            knots,
            np.full(pad_right, knots[-1], dtype=knots.dtype)
        ))
    if n_times == 0:
        return knots
    
    # Find indices where the knot value changes
    jump_idx: npt.NDArray[np.int_] = np.where(knots[1:] > knots[:-1])[0]
    left_vals = knots[jump_idx]
    right_vals =knots[jump_idx + 1]
    num_new_points = (1<<n_times)-1
    fractions = np.linspace(0.,1.,num_new_points+2)[1:-1]
    new_points = left_vals[:, None] + (right_vals - left_vals)[:, None] * fractions[None, :]
    new_points = new_points.ravel()
    insert_positions = np.repeat(jump_idx + 1, num_new_points)
    
    return np.insert(knots, insert_positions, new_points)

p0 = 2
L = 3.
h=2.
n_refinements = 0
knotsx = np.array([0., L/2., L], dtype=np.float64)
knotsx = refine(knotsx, p=p0, n_times=n_refinements)
#log_initial_mesh_size = np.log2(np.max(np.diff(knotsx)))
knotsy = refine(np.array([0., h/2., h], dtype=np.float64), p=p0, n_times=0)
err_cells = {}
hs = HierarchicalSpace(knots=[knotsx, knotsy], degrees=[p0])
        

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=True)
hs.hmesh.plot_cells()

In [ ]:
total_active_cells = sum(len(hs.hmesh.aelem_level[l]) for l in range(hs.nlevels))

all_cells = np.empty((2**hs.dim*total_active_cells, hs.dim), dtype=np.float64) # will have coarser cells on top and finer on bottom
thb_operators: dict[tuple[int, int], npt.NDArray[np.float64]] = {}
N_max = 0 # maximum amount of dofs in a cell
current_idx = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    if len(active_cells_l)==0:
        continue
   
    thb_operators_list = hs.local_multi_level_extraction_operator2(active_cells_l, l, l)
    thb_operators.update({(l, cell): op for cell, op in zip(active_cells_l, thb_operators_list)})
    
    if thb_operators_list:
        level_max = max(op.shape[0] for op in thb_operators_list)
        N_max = max(N_max, level_max)

    mesh = CartesianMesh(hs.hmesh.one_d_indices[l], len(hs.hmesh.one_d_indices[l]))
    my_cells_l = mesh.cells[active_cells_l]

    n_cells = len(my_cells_l)
    x_coords = my_cells_l[:, 0, :]
    y_coords = my_cells_l[:, 1, :] 

    start, end = current_idx, current_idx+(2**hs.dim*n_cells)

    view = all_cells[start:end]
    view[0::4] = np.column_stack((x_coords[:, 0], y_coords[:, 0]))  # (xmin, ymin)
    view[1::4] = np.column_stack((x_coords[:, 1], y_coords[:, 0]))  # (xmax, ymin)
    view[2::4] = np.column_stack((x_coords[:, 0], y_coords[:, 1]))  # (xmin, ymax)
    view[3::4] = np.column_stack((x_coords[:, 1], y_coords[:, 1]))  # (xmax, ymax)
    current_idx=end
pass
# del mesh # free this big object
all_cells = np.array(all_cells).reshape(-1, hs.dim)

coordinates = np.arange(len(all_cells), dtype=np.int32).reshape(-1, 2**hs.dim)
coordinate_element = basix.ufl.element("Q", "quadrilateral", 1, shape=(hs.dim,))
disconnected_mesh = dolfinx.mesh.create_mesh(MPI.COMM_WORLD, cells=coordinates, e=coordinate_element, x=all_cells)


In [ ]:
def up_bottom_boundary(x):
    #on_left = np.isclose(x[0], knots1[0])
    on_bottom = np.isclose(x[1], knotsy[0])
    #on_right = np.isclose(x[0], knots1[-1])
    on_top = np.isclose(x[1], knotsy[-1])
    return on_bottom|on_top

def left_right_boundaries(x):
    on_left = np.isclose(x[0], knotsx[0])
    on_right = np.isclose(x[0], knotsx[-1])
    return on_left|on_right

dim = disconnected_mesh.topology.dim
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)

V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
# print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")
facet_dim = disconnected_mesh.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, up_bottom_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    disconnected_mesh,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)
custom_metadata = {"quadrature_degree": 3}
ds_custom = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=boundary_tags, metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)
#u_sol = dolfinx.fem.Function(V, name="Displacement")

u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
x = ufl.SpatialCoordinate(disconnected_mesh)
u_exact = x[0] * (L - x[0]) * (1.+ x[1])
#u_exact = x[0]*x[1]*(L-x[0])*(h-x[1])
f_mms = -ufl.div(ufl.grad(u_exact))
T_mms = ufl.inner(ufl.grad(u_exact),ufl.FacetNormal(disconnected_mesh))

a = ufl.inner(ufl.grad(u), ufl.grad(v)) * dx_custom
L_cell = ufl.inner(f_mms, v) * dx_custom 
L_facet = ufl.inner(T_mms, v)*ds_custom(1) 


msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_cell, _, _ = ffcx_jit(msh.comm, L_cell, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernel_L_cell = getattr(ufcx_L_cell.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_facet, _, _ = ffcx_jit(msh.comm, L_facet, form_compiler_options={"scalar_type": dtype})
kernel_L_facet = getattr(ufcx_L_facet.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")

ffi = cffi.FFI()

In [ ]:
dofmap, dummy_dof_index = hs.build_global_dof_map()
dummy_dof_index += 1 
num_control_points_with_dummy = dummy_dof_index+1

num_cells = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_global

# which BSpline functions are active on each cell, padded with the dummy index.
padded_cells_to_dofs = np.full((num_cells, N_max), dummy_dof_index, dtype=np.int32)

# Find the maximum local index for each level to size arrays
max_local_indices = [0] * hs.nlevels
for l, local_idx in dofmap.keys():
    if local_idx > max_local_indices[l]:
        max_local_indices[l] = local_idx
    pass
pass

# convert the dictionary into a list of numpy arrays for faster lookup
dofmap_arrays = [np.full(size + 1, dummy_dof_index, dtype=np.int32) for size in max_local_indices]
for (l, local_idx), global_idx in dofmap.items():
    dofmap_arrays[l][local_idx] = global_idx


cell_index = 0
for l in range(hs.nlevels):
    active_cells_l = hs.hmesh.aelem_level[l]
    for cell in active_cells_l:
        
        # Get the local functions on the cell
        active_funcs_dict: dict[int, npt.NDArray[np.int_]] = hs.get_all_active_functions_on_cell(l, cell)
            
        #  Vectorized conversion from local to global indices
        mapped_arrays = [
            dofmap_arrays[ll][local_funcs] for ll, local_funcs in active_funcs_dict.items() if len(local_funcs) > 0
        ]
        
        if mapped_arrays: # Check if there are actually active functions
            global_dofs = np.concatenate(mapped_arrays)
            n_dofs = len(global_dofs)
            padded_cells_to_dofs[cell_index, :n_dofs] = global_dofs
            
        cell_index += 1
new_pctd = padded_cells_to_dofs

M = hs._bezier_to_legendre(degree = p0)
S_indices = np.arange(p0+1, dtype=np.float64)
# scaling for unnormalised Legendre basis polynomials
S_inv = (1./np.sqrt(2.*S_indices+1.))*np.identity(p0+1, dtype=np.float64) # Scaling factor, since fenicsx uses orthonormal legendre polynomials
T = np.asfortranarray(np.kron(M, M).T @ np.kron(S_inv, S_inv), dtype=dtype)
local_dofs_size = T.shape[1]

operator_shape = (N_max, local_dofs_size)
# Create a custom space that holds the content of each matrix for the relevant cell.
# degree 0 because the value is constant over each cell
C_space = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0, operator_shape))
C_func = dolfinx.fem.Function(C_space, dtype=dtype)

num_cells_local = disconnected_mesh.topology.index_map(disconnected_mesh.topology.dim).size_local
indices = np.arange(num_cells_local, dtype=np.int32)
# To make sure that each matrix is assigned to the correct cell
midpoints: npt.NDArray[np.float_] = dolfinx.mesh.compute_midpoints(disconnected_mesh, disconnected_mesh.topology.dim, indices)

c_values = C_func.x.array.reshape((-1, N_max, local_dofs_size))
for local_idx, midpoint in enumerate(midpoints):
    #print(f"midpoint = {midpoint[:2]}")
    level, idx = hs.hmesh.find_active_cell(midpoint[:hs.dim])
    #level, idx = find_level_idx_from_midpoint(hs, midpoint=midpoint[:hs.dim], all_midpoints=physical_cells_midpoints)
    #print(f"level={level}, idx={idx}\n")

    mat = thb_operators[level, idx] @ hs.level_spaces[level].get_bezier_operator(idx)
    
    real_k, n_cols = mat.shape

    if real_k<N_max:
        padding_size = N_max - real_k
        to_pad = np.zeros((padding_size, mat.shape[1]), dtype=np.float64)
        #mat_padded = np.vstack((mat, np.zeros((padding_size, mat.shape[1])) ))
        Ci = mat
    else:
        Ci = mat
        to_pad = np.zeros((0, mat.shape[1]))
    
    # is a view of C_func.x.array, therefore we modify the content of C_func.x.array
    # No new array is created, the matrix->cell mapping is done here.
    c_values[local_idx, :, :] = np.vstack((Ci@T, to_pad))
C_func.x.scatter_forward()

num_control_points = np.max(new_pctd)+1
my_index_map = dolfinx.common.IndexMap(comm=disconnected_mesh.comm, 
                                       local_size=num_control_points)

dummy_element = basix.ufl.element(
    family="DG", 
    cell="quadrilateral", 
    degree=0, 
    shape=(N_max,)
)
dummy_space = dolfinx.fem.functionspace(mesh=disconnected_mesh, 
                                        element=dummy_element)
element_layout = dummy_space.dofmap.dof_layout
cpp_element = dummy_space.element._cpp_object

dof_indices = padded_cells_to_dofs.ravel().astype(np.int32)
offsets = (np.arange(num_cells + 1, dtype=np.int32) * N_max).astype(np.int32)

adj = dolfinx.cpp.graph.AdjacencyList_int32(data=dof_indices, 
                                            offsets=offsets)

doflinx_dofmap = dolfinx.cpp.fem.DofMap(
    element_dof_layout=element_layout, 
    index_map=my_index_map,  
    index_map_bs=1, 
    dofmap=adj, 
    bs=1
)

V_spline_cpp = dolfinx.cpp.fem.FunctionSpace_float64(
    mesh=disconnected_mesh._cpp_object, 
    element=cpp_element, 
    dofmap=doflinx_dofmap
)

V_spline = dolfinx.fem.FunctionSpace(
    mesh=disconnected_mesh, 
    element=dummy_element, 
    cppV=V_spline_cpp
)

facet_dim = msh.topology.dim-1

boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, up_bottom_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for f in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(f)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == f)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
PADDED_DOFS = N_max
LOCAL_DOFS = local_dofs_size

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    # This has to be larger since we are working with padded arrays. Irrelevant dofs are mapped to a dummy location
    A = numba.carray(A_, (PADDED_DOFS, PADDED_DOFS), dtype=dtype)

    # Get the operator (TRUNC @ C_{B->BS} @ (C_{L->B}.T) @ S^{-1}) for this cell
    # TRUNC has shape (PADDED_DOFS, LOCAL_DOFS) and all other matrices have shape (LOCAL_DOFS, LOCAL_DOFS)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)
    
    # Tabulate all sub blocks locally
    # This matrix is formed via the Legendre elements on a quadrilateral of degree p0,
    # therefore this has the shape (LOCAL_DOFS, LOCAL_DOFS)
    A0 = np.zeros((LOCAL_DOFS, LOCAL_DOFS), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,# weights. Ignored for the L2 approximation because 
        # a0 does not depend on f and no value is looked up at the quadrature points
        # This is not very robust, one must be careful when calling this function
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
    
    A[:, :] = G@A0@(G.T) #np.append(np.append(A0, np.zeros((PADDED_DOFS-LOCAL_DOFS, LOCAL_DOFS)), axis=0), 
                       # np.zeros((PADDED_DOFS, PADDED_DOFS-LOCAL_DOFS)), axis=1)
 # 

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_L_cell(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    
    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # evaluation of f using padded THB-Splines
    b = numba.carray(b_, (PADDED_DOFS,), dtype=dtype)

    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)

    b0 = np.zeros((LOCAL_DOFS,), dtype=dtype)
    kernel_L_cell(ffi.from_buffer(b0),
        w_,
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )
   
    b[:] = G@b0

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_L_facet(b1_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    b1 = numba.carray(b1_, (PADDED_DOFS,), dtype=dtype)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)

    b2 = np.zeros((LOCAL_DOFS,), dtype=dtype)
    kernel_L_facet(ffi.from_buffer(b2), 
                   w_, 
                   c_, 
                   coords_, 
                   entity_local_index, 
                   permutation, 
                   empty_void_pointer())
   
    b1[:] = G @ b2

In [ ]:
#cpp_constants = [lmbda_c._cpp_object, mu_c._cpp_object]
cpp_constants = []
formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object
                    ], # weights w_, holds C@T
              constants=cpp_constants,
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.cell: [(0, tabulate_L_cell.address, cells, np.array([0], dtype=np.int8))],
                 dolfinx.fem.IntegralType.exterior_facet: [(0, tabulate_L_facet.address, boundary_entities, np.array([0], dtype=np.int8))]
                 }
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of stuff to integrate
        coefficients=[C_func._cpp_object], # holds C@T
        constants=cpp_constants, need_permutation_data=False, entity_maps=[], mesh=msh._cpp_object
    )
)

In [ ]:
def get_spline_indices_left_right(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        right_dirichlet_indices = np.isclose(basis[:, 1, -hs.degrees[0]-1:], np.full((hs.degrees[0]+1), fill_value=L, dtype=np.float64))
        right_dirichlet_indices = np.all(right_dirichlet_indices, axis=-1)
        right_dirichlet_indices = np.nonzero(right_dirichlet_indices)[0]

        left_dirichlet_indices = np.isclose(basis[:, 1, :-1], np.zeros((hs.degrees[0]+1), dtype=np.float64))
        left_dirichlet_indices = np.all(left_dirichlet_indices, axis=-1)
        left_dirichlet_indices = np.nonzero(left_dirichlet_indices)[0]

        # (A\cap B)\cup(A\cap C) = A\cap(B\cup C)
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  np.union1d(right_dirichlet_indices,left_dirichlet_indices),
                                                  assume_unique=True)
    pass
    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices

def get_spline_indices_all_around(hs, dofmap):
    
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        right_dirichlet_indices = np.isclose(basis[:, 1, -hs.degrees[0]-1:], np.full((hs.degrees[0]+1), fill_value=L, dtype=np.float64))
        right_dirichlet_indices = np.all(right_dirichlet_indices, axis=-1)
        right_dirichlet_indices = np.nonzero(right_dirichlet_indices)[0]

        left_dirichlet_indices = np.isclose(basis[:, 1, :-1], np.zeros((hs.degrees[0]+1), dtype=np.float64))
        left_dirichlet_indices = np.all(left_dirichlet_indices, axis=-1)
        left_dirichlet_indices = np.nonzero(left_dirichlet_indices)[0]

        up_dirichlet_indices = np.isclose(basis[:, 0, -hs.degrees[0]-1:], np.full((hs.degrees[0]+1), fill_value=h, dtype=np.float64))
        up_dirichlet_indices = np.all(up_dirichlet_indices, axis=-1)
        up_dirichlet_indices = np.nonzero(up_dirichlet_indices)[0]

        down_dirichlet_indices = np.isclose(basis[:, 0, :-1], np.zeros((hs.degrees[0]+1), dtype=np.float64))
        down_dirichlet_indices = np.all(down_dirichlet_indices, axis=-1)
        down_dirichlet_indices = np.nonzero(down_dirichlet_indices)[0]

        # (A\cap B)\cup(A\cap C) = A\cap(B\cup C)
        all_dirichlet = np.unique(np.concatenate((right_dirichlet_indices, left_dirichlet_indices, up_dirichlet_indices, down_dirichlet_indices)))
        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  all_dirichlet,
                                                  assume_unique=True)
    pass
    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices

forbidden_indices = get_spline_indices_left_right(hs, dofmap)


In [ ]:
forbidden_indices

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
# a_form = dolfinx.fem.form(a0)
A = assemble_matrix(a_cond, bcs=[])
A.assemble()
one_active=False
two_active= False
for level in range(hs.nlevels):
    if level in hs.truly_active and hs.truly_active[level].size>0:
        if one_active:
            two_active=True
        one_active=True

A_mat = A
if two_active:
    A_mat.setValue(dummy_dof_index, dummy_dof_index, 1., addv=PETSc.InsertMode.INSERT_VALUES)
    A_mat.assemble()
    A_mat.assemblyBegin()
    A_mat.assemblyEnd()

b = assemble_vector(l_cond)
if two_active:
    b[dummy_dof_index]=0.
    b.assemblyBegin()
    b.assemblyEnd()


In [ ]:
if forbidden_indices is not None:
    A_mat.zeroRowsColumns(forbidden_indices, diag=1.0, x=None, b=b)
    b.array_w[forbidden_indices]=0.
b.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)


In [ ]:
print(b.array)

In [ ]:
ksp = PETSc.KSP().create(A_mat.comm)
ksp.setOperators(A_mat)
#ksp.setType(PETSc.KSP.Type.CG)
#ksp.getPC().setType(PETSc.PC.Type.JACOBI)
ksp.setType(PETSc.KSP.Type.PREONLY)
ksp.getPC().setType(PETSc.PC.Type.LU)
ksp.getPC().setFactorSolverType("mumps")
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(b, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
import scipy.sparse as sp
# Extract the CSR (Compressed Sparse Row) arrays from PETSc
indptr, indices, data = A_mat.getValuesCSR()

# Get the global size of the matrix
shape = A_mat.getSize()

# Create a SciPy CSR matrix
A_scipy = sp.csr_array((data, indices, indptr), shape=shape)

print(f"Matrix shape: {A_scipy.shape}")
print(f"Number of non-zeros: {A_scipy.nnz}")

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
# plt.spy plots the non-zero entries of a matrix
plt.spy(A_scipy, markersize=5, color='blue')
plt.title("Sparsity Pattern of THB-Spline Mass Matrix")
plt.show()

In [ ]:
# Map the global B-spline coefficients back to local Legendre coefficients
u_dg = dolfinx.fem.Function(V)

# x_vec is the solution vector
for local_idx in range(msh.topology.index_map(msh.topology.dim).size_local):
    spline_dofs = padded_cells_to_dofs[local_idx]
    u_spline_local = x_vec[spline_dofs]
    
    G = c_values[local_idx, :, :]
    u_dg_local = G.T @ u_spline_local
    
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    u_dg.x.array[dg_dofs] = u_dg_local

u_dg.x.scatter_forward()



# Compute L2 Error: sqrt( \int (u_bar - u_dg)^2 dx )
error_L2_form = dolfinx.fem.form(ufl.inner(u_exact - u_dg, u_exact - u_dg) * dx_custom)
error_L2_sq = dolfinx.fem.assemble_scalar(error_L2_form)
l2_error = np.sqrt(disconnected_mesh.comm.allreduce(error_L2_sq, op=MPI.SUM))

# Compute H1 Semi-norm (Gradient) Error: sqrt( \int |grad(u_bar) - grad(u_dg)|^2 dx )
# This is the "energy" error and is crucial for elliptic PDEs!
error_H1_form = dolfinx.fem.form(ufl.inner(ufl.grad(u_exact) - ufl.grad(u_dg), 
                                           ufl.grad(u_exact) - ufl.grad(u_dg)) * dx_custom)
error_H1_sq = dolfinx.fem.assemble_scalar(error_H1_form)
h1_error = np.sqrt(disconnected_mesh.comm.allreduce(error_H1_sq, op=MPI.SUM))

# Compute Exact Norms (for Relative Error calculations)
norm_L2_form = dolfinx.fem.form(ufl.inner(u_exact, u_exact) * ufl.dx)
exact_L2_norm = np.sqrt(disconnected_mesh.comm.allreduce(dolfinx.fem.assemble_scalar(norm_L2_form), op=MPI.SUM))

norm_H1_form = dolfinx.fem.form(ufl.inner(ufl.grad(u_exact), ufl.grad(u_exact)) * ufl.dx)
exact_H1_norm = np.sqrt(disconnected_mesh.comm.allreduce(dolfinx.fem.assemble_scalar(norm_H1_form), op=MPI.SUM))

# Print results
print(f"Absolute L2 Error: {l2_error:.2e}")
print(f"Relative L2 Error: {l2_error / exact_L2_norm:.2e}\n")

print(f"Absolute H1 Error: {h1_error:.2e}")
print(f"Relative H1 Error: {h1_error / exact_H1_norm:.2e}")
print(f"dofs = {A_mat.getSize()[0]-1}")

In [ ]:
# b.array

In [ ]:
import dolfinx.plot
import pyvista

# 1. Create a "Nodal" DG space of the same degree for plotting
# By default, DG with no variant specified uses Lagrange (nodal)
v_plot_elt = basix.ufl.element(
    "DG", 
    "quadrilateral", 
    degree=p0+2
)
V_plot = dolfinx.fem.functionspace(disconnected_mesh, v_plot_elt)

# 2. Interpolate your computed solution (u_dg) into the nodal space
u_plot = dolfinx.fem.Function(V_plot)
#error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-5)
error_ufl = u_dg
error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
u_error = dolfinx.fem.Function(V_plot)
u_error.interpolate(error_expr)

# 3. Now use V_plot for the VTK mesh generation
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
grid.point_data["u"] = u_error.x.array.real
grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
plotter = pyvista.Plotter()
grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
plotter.view_xy()
plotter.show(jupyter_backend="static")
plotter.show()